# Laboratorio 2 - Deep Learning

**Integrantes:**

- Flavio Galán
- Josue Say

**Repositorio:**

- [Enlace GitHub](https://github.com/JosueSay/labs-ds/tree/main/lab2)

### Librerías

In [16]:

import pandas as pd
import numpy as np
import pyreadr
import matplotlib.pyplot as plt
import statsmodels.tsa as tsa
import statsmodels as sm
from datetime import datetime
import os
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import pmdarima as pm
from pmdarima.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tabulate import tabulate
from prophet import Prophet
from statsmodels.tsa.holtwinters import ExponentialSmoothing

### Variables - constantes

In [17]:
DATA_DIR = "../data"

REPORTES = "./reportes"
CACHE = "./cache"

# output files
CONSUMO_CSV = f"{DATA_DIR}/consumo_combustibles.csv"
IMPORT_CSV = f"{DATA_DIR}/importacion_combustibles.csv"
PRECIOS_CSV = f"{DATA_DIR}/precios_diarios.csv"
TARGET_COLUMNS = ["fecha", "regular", "superior", "diesel", "glp"]

### Obtención de data

In [18]:
def loadAllData():
    df_consumo = pd.read_csv(CONSUMO_CSV, parse_dates=["fecha"])
    df_importaciones = pd.read_csv(IMPORT_CSV, parse_dates=["fecha"])
    df_precios = pd.read_csv(PRECIOS_CSV, parse_dates=["fecha"])

    print("Datos cargados desde cache CSV.")
    return df_consumo, df_importaciones, df_precios

In [19]:
df_consumo, df_importaciones, df_precios = loadAllData()

Datos cargados desde cache CSV.


## Modelos LSTM para serie de "Precios de Gasolina Regular"

### Constantes

In [20]:
TRAIN_FRACTION = 0.6
VAL_FRACTION = 0.2
TEST_FRACTION = 0.2

In [21]:
def plotSeries(df, column, title):
    plt.figure(figsize=(10, 5))
    df[column].plot()
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [22]:
def decomposeSeries(series):
    result = seasonal_decompose(series)
    fig = result.plot()
    fig.set_size_inches(15, 8)
    plt.tight_layout()
    plt.show()

In [23]:
def plotAutocorrelations(series):
    plot_acf(series)
    plt.tight_layout()
    plt.show()
    plot_pacf(series)
    plt.tight_layout()
    plt.show()

In [24]:
def dickeyFullerTest(series):
    print('ADF Test Results')
    dfTest = adfuller(series, autolag='AIC')
    output = pd.Series(dfTest[0:4], index=['Test Statistic','p-value','Lags Used','N Observations'])
    for key, value in dfTest[4].items():
        output[f'Critical Value ({key})'] = value
    print(output)

In [25]:
def splitSeries(series):
    train_size = int(len(series) * TRAIN_FRACTION)
    val_size = int(len(series) * VAL_FRACTION)
    test_size = len(series) - train_size - val_size
    train = series[:train_size]
    val = series[train_size:train_size + val_size]
    test = series[train_size + val_size:]
    return train, val, test

In [26]:
def trainARIMAModels(trainData):
    models = [
        ARIMA(trainData, order=(1,1,1)),
        ARIMA(trainData, order=(0,1,1)),
        ARIMA(trainData, order=(1,1,0))
    ]
    return [model.fit() for model in models]

In [27]:
def plotResiduals(model):
    residuals = model.resid[1:]
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    residuals.plot(title="Residuals", ax=ax[0])
    residuals.plot(title="Density", kind="kde", ax=ax[1])
    plt.tight_layout()
    plt.show()

In [28]:
def evaluateARIMAModels(models, testData):
    newDF = pd.DataFrame({'Original': testData})
    newDF = newDF.assign(
        Model_1_1_1=lambda _: list(models[0].forecast(len(testData))),
        Model_0_1_1=lambda _: list(models[1].forecast(len(testData))),
        Model_1_1_0=lambda _: list(models[2].forecast(len(testData))),
        Model_auto=lambda _: list(pm.auto_arima(models[0].data.endog, seasonal=False).predict(len(testData)))
    )
    newDF.plot()
    plt.title("ARIMA Model Predictions")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [29]:
def runProphetModel(trainData):
    df_train = pd.DataFrame({"ds": trainData.index, "y": list(trainData)})
    model = Prophet()
    model.fit(df_train)
    future = model.make_future_dataframe(periods=365)
    forecast = model.predict(future)
    model.plot(forecast)
    model.plot_components(forecast)

In [30]:
def runHoltWintersModel(trainData):
    model = ExponentialSmoothing(trainData, trend='add', seasonal='add', seasonal_periods=12)
    fit = model.fit()
    forecast_steps = 6
    predictions = fit.forecast(forecast_steps)
    return predictions

In [31]:
def convertToSupervised(series, nLags=1):
    x, y = [], []
    for i in range(len(series) - nLags):
        x.append(series[i:i + nLags, 0])
        y.append(series[i + nLags, 0])
    return np.array(x), np.array(y)

In [32]:
def runMLPModel(scaledSeries):
    n = len(scaledSeries)
    train_size = int(n * TRAIN_FRACTION)
    val_size = int(n * VAL_FRACTION)

    train = scaledSeries[:train_size]
    val = scaledSeries[train_size:train_size + val_size]
    test = scaledSeries[train_size + val_size:]

    x_train, y_train = convertToSupervised(train, 5)
    x_val, y_val = convertToSupervised(val, 5)
    x_test, y_test = convertToSupervised(test, 5)

    model = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', max_iter=1000, early_stopping=True, random_state=1)
    model.fit(x_train, y_train)

    pred_test = model.predict(x_test)
    return pred_test, y_test

In [33]:
def inverseScaleAndEvaluate(predicted, actual, scaler):
    pred_inv = scaler.inverse_transform(np.array(predicted).reshape(-1,1))
    actual_inv = scaler.inverse_transform(np.array(actual).reshape(-1,1))
    rmse = np.sqrt(mean_squared_error(actual_inv, pred_inv))
    mae = mean_absolute_error(actual_inv, pred_inv)
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")

    plt.figure(figsize=(10,5))
    plt.plot(actual_inv, label='Actual')
    plt.plot(pred_inv, label='Predicted', alpha=0.7)
    plt.title("MLP - Test Set Prediction")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [34]:

# === Ejemplo de uso secuencial ===
# df_rango = df_precios["2022":]
# plotSeries(df_rango, "regular", "Precios - regular (2022-2025)")
# decomposeSeries(df_rango["regular"])
# plotAutocorrelations(df_rango["regular"])
# dickeyFullerTest(df_rango["regular"])
# ts_diff = df_rango["regular"].diff().fillna(0)
# dickeyFullerTest(ts_diff)
# train_data, val_data, test_data = splitSeries(ts_diff)
# arima_models = trainARIMAModels(train_data)
# for model in arima_models:
#     plotResiduals(model)
# evaluateARIMAModels(arima_models, test_data)
# runProphetModel(train_data)
# runHoltWintersModel(train_data)

# scaler = StandardScaler()
# ts_diff_scaled = scaler.fit_transform(ts_diff.to_frame())
# pred_test, y_test = runMLPModel(ts_diff_scaled)
# inverseScaleAndEvaluate(pred_test, y_test, scaler)

## OTRA SERIE